In [ ]:
library(Seurat)
library(readr)
library(dplyr)
library(ggplot2)
library(pheatmap)
library(tidyr) 
library(patchwork)
library(tidyverse)
library(ggrepel)
library(scales)

In [ ]:
## 皮层单细胞可视化

In [ ]:
data <- readRDS("data/processed/cortex_analysis_file")
data

In [ ]:
table(data@meta.data$annotation)

In [ ]:
# 大类配色 NASA_cortex_color.csv

In [ ]:
cluster_colours <- c(
  Astro = "#6baed6",
  Choroid_plexus = "#3182bd",
  Endo = "#08519c",
  Ep = "#31a354",
  Ex = "#74c476",
  Fibroblast = "#a1d99b",
  Immune = "#e6550d",
  In = "#ff851b",
  Oligo = "#fee6cd",
  Opc = "#fdae6b"
)

In [ ]:
# 提取UMAP坐标和元数据
umap_data <- data@reductions$umap@cell.embeddings
meta_data <- data@meta.data

# 将UMAP坐标和元数据合并为一个数据框
df <- as_tibble(umap_data, rownames = 'cells') %>%
  left_join(., meta_data %>% rownames_to_column(var = "cells")) %>%
  select(cells, UMAP_1, UMAP_2, annotation) %>%
  group_by(annotation) %>%
  mutate(avg_umap_1 = mean(UMAP_1), avg_umap_2 = mean(UMAP_2)) %>%
  ungroup()

In [ ]:
# 绘制UMAP图
p <- ggplot(df %>% slice_sample(n = nrow(df))) +
  geom_point(mapping = aes(UMAP_1, UMAP_2, color = annotation), size = 0.1, alpha = 0.6) +
  theme_void() +
  theme(legend.position = 'none') +
  scale_color_manual(values = cluster_colours) +
  coord_fixed(ratio = 0.7)

# 显示图形
print(p)

# 保存图像为PNG和PDF
output_png_file <- "results/figures/figure_panel_file"
output_pdf_file <- "results/figures/figure_panel_file"

ggsave(output_png_file, plot = p, width = 6, height = 8, units = "in")
ggsave(output_pdf_file, plot = p, width = 6, height = 8, units = "in")

cat("UMAP plot saved to", output_png_file, "\n")
cat("UMAP plot saved to", output_pdf_file, "\n")

In [ ]:
## 差异基因气泡图，按ratio过滤

In [ ]:
Idents(data) <- data@meta.data$annotation
markers <- FindAllMarkers(data, only.pos = TRUE, min.pct = 0.25, logfc.threshold = 0.25)
write.csv(markers, file = "data/processed/cortex_analysis_file", row.names = FALSE)

In [ ]:
markers <- read.csv("data/processed/cortex_analysis_file")

# 检查是否存在pct.1和pct.2列
if("pct.1" %in% colnames(markers) && "pct.2" %in% colnames(markers)) {
  # 添加ratio列
  markers$ratio <- (markers$pct.1 - markers$pct.2) / markers$pct.1
} else {
  stop("pct.1 或 pct.2 列不存在，请检查数据！")
}

markers_filtered <- markers[markers$ratio > 0.5, ]
head(markers_filtered)
write.csv(markers_filtered, file = "data/processed/cortex_analysis_file", row.names = FALSE)

In [ ]:
markers <- read.csv("data/processed/cortex_analysis_file")

# 获取每个cluster排名前5的gene
top_genes <- markers %>%
  group_by(cluster) %>%
  slice_head(n = 5) %>%
  ungroup()

# 提取需要绘制的gene
genes_to_plot <- unique(top_genes$gene)

In [ ]:
# 绘制点图
dotplot <- DotPlot(data, features = genes_to_plot, group.by = "annotation", 
                   dot.scale = 6,cols=c("#1F66AC","#B2192B"),col.min = -1,col.max = 2) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  theme(plot.title = element_blank()) + # 去除标题
  theme(axis.title.x = element_blank(), # 去除横坐标轴标题
        axis.title.y = element_blank()) # 去除纵坐标轴标题

ggsave("results/figures/figure_panel_file", plot = dotplot, width = 15, height = 8)
ggsave("results/figures/figure_panel_file", plot = dotplot, width = 15, height = 8)

In [ ]:
## 绘制堆叠柱状图

In [ ]:
merged_data <- readRDS("data/processed/cortex_analysis_file")
merged_data

In [ ]:
# 设置Idents
Idents(merged_data) <- merged_data@meta.data$annotation_c
table(merged_data@meta.data$annotation_c)

In [ ]:
meta_data <- merged_data@meta.data
meta_data$className <- gsub("^(.+)_.*", "\\1", meta_data$annotation_c)
cell_counts <- table(meta_data$className, gsub("^.*_(.*)$", "\\1", meta_data$annotation_c))
df <- as.data.frame(cell_counts)
colnames(df) <- c("className", "rlCodes", "n")
df <- df[df$className %in% c("hm", "nasa") & df$rlCodes != "other", ]
df <- df[, c("className", "rlCodes", "n")]
print(df)

In [ ]:
df %>% 
  group_by(className) %>% 
  summarise(total_number=sum(n)) %>% 
  ungroup() %>% 
  mutate(ratio=total_number/sum(total_number)) %>% 
  mutate(ratio=scales::percent(ratio)) -> df2
print(df2)

In [ ]:
# 计算每个物种中每个细胞类型的比例
df <- df %>%
  left_join(df2, by = "className") %>%
  mutate(ratio = scales::percent(n / total_number))

# 打印结果
print(df)

In [ ]:
# 调整顺序
df$className<-factor(
  df$className,
  levels = c("nasa","hm")
)

df$rlCodes<-factor(
  df$rlCodes,
  levels = rev(c("Excit","Inhib","Oligo","OPC","Astro","EndoMural","Micro")))

ggplot(data = df,aes(x=className,y=n,fill=rlCodes))+
  geom_bar(stat = "identity",
           position = "stack")+
  scale_fill_discrete(limits=c("Excit","Inhib","Oligo","OPC","Astro","EndoMural","Micro"))

In [ ]:
# 绘制堆叠柱状图
p <- ggplot(data = df, aes(x = className, y = n, fill = rlCodes)) +
  geom_bar(stat = "identity", position = "fill") +
  scale_fill_manual(values = c("Astro" = "#6baed6", "EndoMural" = "#08519c",
                               "Excit" = "#74c476", "Inhib" = "#ff851b",
                               "Micro" = "#e6550d", "Oligo" = "#fee6cd", "OPC" = "#fdae6b"),
                    limits = c("Excit", "Inhib", "Oligo", "OPC", "Astro", "EndoMural", "Micro")) +
  geom_text(data = df2,
            aes(x = className, y = 1,
                label = paste0(total_number, "\n", "(", ratio, ")")),
            inherit.aes = FALSE,
            vjust = -0.2) +
  scale_y_continuous(expand = expansion(mult = c(0.01, 0.1)),
                     labels = scales::percent_format())+
  theme(panel.background = element_blank(),
        axis.line = element_line(),
        legend.position = "bottom") +
  labs(x = NULL, y = "Species threatened (%)") +
  guides(fill = guide_legend(title = NULL, nrow = 1, byrow = FALSE))

ggsave("results/figures/figure_panel_file", plot = p, width = 6, height = 10, dpi = 300)
ggsave("results/figures/figure_panel_file", plot = p, width = 6, height = 10)

In [ ]:
## 皮层兴奋性神经元可视化

In [ ]:
data <- readRDS("data/processed/excitatory_neuron_file")
#data <- subset(data, subset = annotation != "Other")
data

In [ ]:
table(data@meta.data$annotation)

In [ ]:
# 自定义颜色映射 nasa_combined_exn_color.csv

In [ ]:
cluster_colours <- c(
  EX_ADRA1A_Vat1l = "#E69F00",   # Nature Communications橙色
  EX_BDNF = "#56B4E9",           # Cell Reports蓝色
  EX_C1QL3 = "#009E73",          # Science绿色
  EX_ERG = "#0072B2",            # Nature深蓝
  EX_FOXP2 = "#D55E00",          # Immunity橙红
  EX_HTR7_GNAL = "#CC79A7",      # Nature Methods紫红
  EX_IFITM3_CLDN5 = "#332288",   # Nature Neuroscience深紫
  EX_ITGA4_TPBG = "#88CCEE",     # Science Advances浅蓝
  EX_NTNG1 = "#AA4499",          # Cell Metabolism紫色
  EX_Ntng2 = "#117733",          # Nature Medicine深绿
  EX_Nxph4_CPLX3 = "#661100",    # Cell Reports棕红
  EX_SLC1A3_FGFR3 = "#999933",   # Science Signaling橄榄绿
  EX_TSHZ2_VWC2L = "#DDCC77",    # Nature Genetics米黄
  EX_env_TMEM144 = "#44AA99"      # Neuron蓝绿色
)

# 提取UMAP坐标和元数据
umap_data <- data@reductions$umap@cell.embeddings
meta_data <- data@meta.data

# 将UMAP坐标和元数据合并为一个数据框
df <- as_tibble(umap_data, rownames = 'cells') %>%
  left_join(., meta_data %>% rownames_to_column(var = "cells")) %>%
  select(cells, UMAP_1, UMAP_2, annotation) %>%
  group_by(annotation) %>%
  mutate(avg_umap_1 = mean(UMAP_1), avg_umap_2 = mean(UMAP_2)) %>%
  ungroup()

In [ ]:
# 绘制UMAP图
p <- ggplot(df %>% slice_sample(n = nrow(df))) +
  geom_point(mapping = aes(UMAP_1, UMAP_2, color = annotation), size = 0.1, alpha = 0.6) +
  theme_void() +
  theme(legend.position = 'none') +
  scale_color_manual(values = cluster_colours) +
  coord_fixed(ratio = 0.7)

# 显示图形
print(p)

# 保存图像为PNG和PDF
output_png_file <- "results/figures/figure_panel_file"
output_pdf_file <- "results/figures/figure_panel_file"

ggsave(output_png_file, plot = p, width = 6, height = 8, units = "in")
ggsave(output_pdf_file, plot = p, width = 6, height = 8, units = "in")

cat("UMAP plot saved to", output_png_file, "\n")
cat("UMAP plot saved to", output_pdf_file, "\n")

In [ ]:
## 差异基因气泡图，按ratio过滤

In [ ]:
Idents(data) <- "annotation"
markers <- FindAllMarkers(data, only.pos = TRUE, min.pct = 0.25, logfc.threshold = 0.25)
write.csv(markers, file = "data/processed/excitatory_neuron_file", row.names = FALSE)

In [ ]:
markers <- read.csv("data/processed/excitatory_neuron_file")

# 检查是否存在pct.1和pct.2列
if("pct.1" %in% colnames(markers) && "pct.2" %in% colnames(markers)) {
  # 添加ratio列
  markers$ratio <- (markers$pct.1 - markers$pct.2) / markers$pct.1
} else {
  stop("pct.1 或 pct.2 列不存在，请检查数据！")
}

markers_filtered <- markers[markers$ratio > 0.5, ]
head(markers_filtered)
write.csv(markers_filtered, file = "data/processed/excitatory_neuron_file", row.names = FALSE)

In [ ]:
markers <- read.csv("data/processed/excitatory_neuron_file")

# 获取每个cluster排名前5的gene
top_genes <- markers %>%
  group_by(cluster) %>%
  slice_head(n = 5) %>%
  ungroup()

# 提取需要绘制的gene
genes_to_plot <- unique(top_genes$gene)

In [ ]:
# 绘制点图
dotplot <- DotPlot(data, features = genes_to_plot, group.by = "annotation", 
                   dot.scale = 6,cols=c("#1F66AC","#B2192B"),col.min = -1,col.max = 2) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  theme(plot.title = element_blank()) + # 去除标题
  theme(axis.title.x = element_blank(), # 去除横坐标轴标题
        axis.title.y = element_blank()) # 去除纵坐标轴标题

ggsave("results/figures/figure_panel_file", plot = dotplot, width = 15, height = 8)
ggsave("results/figures/figure_panel_file", plot = dotplot, width = 15, height = 8)